# Sales and Profit Analysis with Machine Learning

**Mata Kuliah:** Artificial Intelligence  
**Dataset:** `Ultimate_Sales_Profit_Dataset.xlsx`

## Tujuan
Proyek ini bertujuan memahami pola penjualan dan keuntungan, kemudian membangun model Machine Learning sederhana untuk memperkirakan nilai profit berdasarkan informasi transaksi.

Metode yang digunakan adalah *Exploratory Data Analysis* (EDA) dan **Random Forest Regressor**. Model dievaluasi menggunakan MAE, RMSE, dan R².


## 1. Import Library

Pandas dan NumPy digunakan untuk pengolahan data. Matplotlib dan Seaborn digunakan untuk visualisasi, sedangkan Scikit-learn digunakan untuk membangun dan mengevaluasi model.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)


## 2. Memuat Dataset

Dataset Excel dibaca dari folder `data`.

In [ ]:
df = pd.read_excel("../data/Ultimate_Sales_Profit_Dataset.xlsx")

print("Ukuran dataset:", df.shape)
display(df.head())


## 3. Memahami Dataset

Melihat informasi kolom dan statistik deskriptif untuk mengenali data transaksi.

In [ ]:
df.info()
display(df.describe(include="all").T)


## 4. Pembersihan Data

Memeriksa data kosong dan duplikat. Kolom tanggal dikonversi menjadi format datetime, sedangkan kolom numerik dipastikan memiliki tipe angka.

In [ ]:
print("Nilai kosong:")
display(df.isnull().sum().sort_values(ascending=False).head(15))

print("Jumlah baris duplikat:", df.duplicated().sum())

df = df.drop_duplicates().copy()
df["Order Date"] = pd.to_datetime(df["Order Date"], errors="coerce")

numeric_cols = ["Quantity", "Unit Price", "Discount %", "Cost Price",
                "Sales", "Profit", "Profit Margin %"]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna(subset=["Profit", "Quantity", "Unit Price", "Cost Price"])
print("Ukuran data setelah pembersihan:", df.shape)


## 5. Analisis Statistik Penjualan dan Profit

Menampilkan nilai total penjualan, total profit, rata-rata profit, dan jumlah transaksi.

In [ ]:
print("Total penjualan:", round(df["Sales"].sum(), 2))
print("Total profit:", round(df["Profit"].sum(), 2))
print("Rata-rata profit per transaksi:", round(df["Profit"].mean(), 2))
print("Jumlah transaksi:", len(df))


## 6. Penjualan Berdasarkan Kategori Produk

Grafik ini membandingkan total penjualan pada setiap kategori produk.

In [ ]:
sales_category = df.groupby("Category")["Sales"].sum().sort_values(ascending=False)

plt.figure()
sales_category.plot(kind="bar")
plt.title("Total Penjualan Berdasarkan Kategori")
plt.xlabel("Kategori Produk")
plt.ylabel("Total Sales")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()


## 7. Profit Berdasarkan Wilayah

Mengelompokkan profit berdasarkan region untuk melihat kontribusi keuntungan dari setiap wilayah.

In [ ]:
profit_region = df.groupby("Region")["Profit"].sum().sort_values(ascending=False)

plt.figure()
profit_region.plot(kind="bar")
plt.title("Total Profit Berdasarkan Wilayah")
plt.xlabel("Wilayah")
plt.ylabel("Total Profit")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()


## 8. Tren Penjualan Bulanan

Tanggal transaksi dikelompokkan berdasarkan bulan. Grafik digunakan untuk melihat perubahan total penjualan dari waktu ke waktu.

In [ ]:
df["Month Period"] = df["Order Date"].dt.to_period("M")
monthly_sales = df.groupby("Month Period")["Sales"].sum()

plt.figure(figsize=(12, 5))
monthly_sales.plot(marker="o")
plt.title("Tren Penjualan Bulanan")
plt.xlabel("Bulan")
plt.ylabel("Total Sales")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


## 9. Menyiapkan Data untuk Machine Learning

Target yang akan diprediksi adalah `Profit`. Fitur dipilih dari informasi transaksi yang tersedia. Kolom seperti `Sales`, `Gross Profit`, `Net Profit`, dan `Profit Margin %` tidak digunakan sebagai fitur karena berhubungan langsung dengan perhitungan atau informasi target dan berisiko menyebabkan *data leakage*.

Kolom identitas seperti nomor pesanan dan nama pelanggan juga tidak digunakan sebagai fitur.

In [ ]:
target = "Profit"

feature_cols = [
    "Quantity", "Unit Price", "Discount %", "Cost Price",
    "Category", "Sub Category", "Brand", "Region",
    "Payment Mode", "Ship Mode", "Order Status"
]

X = df[feature_cols].copy()
y = df[target].copy()

numeric_features = ["Quantity", "Unit Price", "Discount %", "Cost Price"]
categorical_features = [col for col in feature_cols if col not in numeric_features]

print("Jumlah fitur:", len(feature_cols))
print("Jumlah data:", len(X))


## 10. Membagi Data Training dan Testing

Data dibagi menjadi 80% data training dan 20% data testing. Data testing digunakan untuk mengevaluasi kemampuan model pada data yang tidak digunakan selama pelatihan.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Data training:", X_train.shape)
print("Data testing:", X_test.shape)


## 11. Membuat Pipeline dan Melatih Model

Nilai numerik yang kosong diisi menggunakan median. Nilai kategorikal yang kosong diisi menggunakan modus, kemudian diubah dengan One-Hot Encoding. Random Forest Regressor digunakan untuk mempelajari pola hubungan antara fitur transaksi dan profit.

In [ ]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        max_depth=None
    ))
])

model.fit(X_train, y_train)
print("Model berhasil dilatih.")


## 12. Evaluasi Model

Prediksi pada data testing dibandingkan dengan nilai profit sebenarnya. MAE dan RMSE mengukur besar kesalahan prediksi, sedangkan R² menunjukkan seberapa besar variasi target yang dapat dijelaskan model.

In [ ]:
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("MAE :", round(mae, 2))
print("RMSE:", round(rmse, 2))
print("R²  :", round(r2, 4))


## 13. Perbandingan Nilai Aktual dan Prediksi

Grafik berikut membandingkan profit aktual dengan hasil prediksi model pada data testing.

In [ ]:
hasil = pd.DataFrame({
    "Profit Aktual": y_test.values,
    "Profit Prediksi": y_pred
})

display(hasil.head(10))

plt.figure()
plt.scatter(hasil["Profit Aktual"], hasil["Profit Prediksi"], alpha=0.7)
batas_min = min(hasil.min())
batas_max = max(hasil.max())
plt.plot([batas_min, batas_max], [batas_min, batas_max], linestyle="--")
plt.title("Profit Aktual vs Profit Prediksi")
plt.xlabel("Profit Aktual")
plt.ylabel("Profit Prediksi")
plt.tight_layout()
plt.show()


## 14. Kesimpulan

Kesimpulan akhir diisi berdasarkan hasil eksekusi notebook. Bahas kategori dengan penjualan tertinggi, wilayah dengan profit terbesar, pola penjualan bulanan, serta nilai MAE, RMSE, dan R² model.

**Catatan:** Dataset berisi 500 transaksi. Hasil evaluasi menggambarkan performa model pada pembagian data yang digunakan dan belum menjamin performa yang sama pada transaksi baru. Model ini merupakan contoh penerapan regresi untuk pembelajaran, bukan sistem prediksi keuangan operasional.
